# Interactive Moxon Rectangle Calculator

This notebook calculates and visualizes practical first-cut dimensions for a Moxon rectangle antenna, with presets for the 2 m and 70 cm amateur radio bands.

The geometry is intended for kit planning and visualization. Final dimensions should be validated by measurement or NEC modeling, especially at VHF/UHF where millimeters matter.

## Theory

In [ ]:
from IPython.display import Markdown
Markdown(filename="moxon_theory.md")

## Calculator Model

The calculator below uses wavelength-scaled starting proportions that match the typical 2 m and 70 cm Moxon calculator examples in this repository. The conductor diameter control applies a small empirical correction to the end gap and folded tails because thicker conductors are electrically shorter and usually require slightly different coupling geometry.

For the build kit H-frame, edit the global offset constants in the next code cell. The four equal outer pipes are calculated as `A / 2 - RADIATOR_PIPE_OFFSET_MM`; the center pipe is calculated as `E - CENTER_PIPE_OFFSET_MM`.

In [ ]:
import math

import matplotlib.pyplot as plt
from IPython.display import HTML, display
import ipywidgets as widgets

# speed of light m/s
C_M_PER_S = 299_792_458

# Practical starter proportions, calibrated to the 145 MHz and 435 MHz calculator images in img/.
BASE_RATIOS = {
    'A_width': 0.361,
    'B_driven_tail': 0.051,
    'C_end_gap': 0.014,
    'D_reflector_tail': 0.069,
    'E_spacing': 0.133,
}

# Build-kit H-frame offsets. Set these once for the printed T-pieces/socket depths.
RADIATOR_PIPE_OFFSET_MM = 24.49
CENTER_PIPE_OFFSET_MM = 54.4914+7.5

def moxon_dimensions(frequency_mhz, conductor_diameter_mm=1.0, velocity_factor=1.0):
    wavelength_m = (C_M_PER_S / (frequency_mhz * 1_000_000)) * velocity_factor
    diameter_ratio = max(conductor_diameter_mm / 1.0, 0.05)
    thick_wire_shift = math.log10(diameter_ratio)

    ratios = dict(BASE_RATIOS)
    ratios['C_end_gap'] = max(0.009, ratios['C_end_gap'] * (1.0 + 0.10 * thick_wire_shift))
    ratios['B_driven_tail'] = max(0.040, ratios['B_driven_tail'] * (1.0 - 0.04 * thick_wire_shift))
    ratios['D_reflector_tail'] = max(0.058, ratios['D_reflector_tail'] * (1.0 - 0.04 * thick_wire_shift))

    dimensions = {name: ratio * wavelength_m for name, ratio in ratios.items()}
    dimensions['wavelength'] = wavelength_m
    dimensions['driven_wire_half'] = dimensions['A_width'] / 2 + dimensions['B_driven_tail']
    dimensions['driven_total'] = dimensions['A_width'] + 2 * dimensions['B_driven_tail']
    dimensions['reflector_wire'] = dimensions['A_width'] + 2 * dimensions['D_reflector_tail']
    dimensions['reflector_total'] = dimensions['A_width'] + 2 * dimensions['D_reflector_tail']
    dimensions['total_build_wire'] = dimensions['driven_total'] + dimensions['reflector_wire']
    dimensions['diagonal'] = math.hypot(dimensions['A_width'], dimensions['E_spacing'])
    dimensions['radiator_pipe_offset'] = RADIATOR_PIPE_OFFSET_MM / 1000
    dimensions['center_pipe_offset'] = CENTER_PIPE_OFFSET_MM / 1000
    dimensions['h_frame_outer_pipe'] = dimensions['A_width'] / 2 - dimensions['radiator_pipe_offset']
    dimensions['h_frame_center_pipe'] = dimensions['E_spacing'] - dimensions['center_pipe_offset']
    return dimensions

def unit_scale(units):
    if units == 'm':
        return 1.0, 'm'
    if units == 'cm':
        return 100.0, 'cm'
    if units == 'mm':
        return 1000.0, 'mm'
    raise ValueError(f'Unsupported unit: {units}')

def format_geometry_table(dimensions, units):
    scale, label = unit_scale(units)
    rows = [
        ('A', 'Driven-side horizontal span', dimensions['A_width']),
        ('B', 'Driven folded tail, each side', dimensions['B_driven_tail']),
        ('C', 'End gap between tails', dimensions['C_end_gap']),
        ('D', 'Reflector folded tail, each side', dimensions['D_reflector_tail']),
        ('E', 'Driven to reflector spacing', dimensions['E_spacing']),
        ('-', 'Corner-to-corner diagonal', dimensions['diagonal']),
    ]
    html_rows = ''.join(
        f'<tr><td><b>{symbol}</b></td><td>{name}</td><td style="text-align:right">{value * scale:.1f} {label}</td></tr>'
        for symbol, name, value in rows
    )
    return HTML(
        '<h3>Electrical geometry</h3>'
        '<table>'
        '<thead><tr><th>Symbol</th><th>Dimension</th><th>Length</th></tr></thead>'
        f'<tbody>{html_rows}</tbody>'
        '</table>'
    )

def format_build_table(dimensions, units):
    scale, label = unit_scale(units)
    rows = [
        ('Pipe', 'Outer H-frame pipe', '4', dimensions['h_frame_outer_pipe'], 4 * dimensions['h_frame_outer_pipe']),
        ('Pipe', 'Center spacing pipe', '1', dimensions['h_frame_center_pipe'], dimensions['h_frame_center_pipe']),
        ('Wire', 'Driven element wire half', '2', dimensions['driven_wire_half'], dimensions['driven_total']),
        ('Wire', 'Reflector wire', '1', dimensions['reflector_wire'], dimensions['reflector_wire']),
        ('Wire', 'Total antenna wire required', '-', dimensions['total_build_wire'], dimensions['total_build_wire']),
        ('Offset', 'Radiator pipe offset', '-', dimensions['radiator_pipe_offset'], dimensions['radiator_pipe_offset']),
        ('Offset', 'Center pipe offset', '-', dimensions['center_pipe_offset'], dimensions['center_pipe_offset']),
    ]
    html_rows = ''.join(
        '<tr>'
        f'<td>{category}</td>'
        f'<td>{part}</td>'
        f'<td style="text-align:right">{quantity}</td>'
        f'<td style="text-align:right">{cut_length * scale:.1f} {label}</td>'
        f'<td style="text-align:right">{total_length * scale:.1f} {label}</td>'
        '</tr>'
        for category, part, quantity, cut_length, total_length in rows
    )
    return HTML(
        '<h3>Build cut lengths</h3>'
        '<table>'
        '<thead><tr><th>Category</th><th>Part</th><th>Quantity</th><th>Cut length</th><th>Total length</th></tr></thead>'
        f'<tbody>{html_rows}</tbody>'
        '</table>'
    )

def plot_moxon(dimensions, frequency_mhz, units):
    scale, label = unit_scale(units)
    A = dimensions['A_width'] * scale
    B = dimensions['B_driven_tail'] * scale
    C = dimensions['C_end_gap'] * scale
    D = dimensions['D_reflector_tail'] * scale
    E = dimensions['E_spacing'] * scale
    outer_pipe = dimensions['h_frame_outer_pipe'] * scale
    center_pipe = dimensions['h_frame_center_pipe'] * scale
    feed_gap = max(A * 0.035, 2.0 if units == 'mm' else 0.2 if units == 'cm' else 0.002)
    mid = A / 2

    fig, ax = plt.subplots(figsize=(11, 6.5), dpi=160)
    lw = 5.5
    color = '#3f3f3f'
    label_box = dict(boxstyle='round,pad=0.2', facecolor='white', edgecolor='none', alpha=0.9)

    # Driven element: split at feed point and folded down at the ends.
    ax.plot([0, mid - feed_gap / 2], [E, E], color=color, lw=lw, solid_capstyle='round')
    ax.plot([mid + feed_gap / 2, A], [E, E], color=color, lw=lw, solid_capstyle='round')
    ax.plot([0, 0], [E, E - B], color=color, lw=lw, solid_capstyle='round')
    ax.plot([A, A], [E, E - B], color=color, lw=lw, solid_capstyle='round')

    # Reflector: continuous bottom side and folded up tails.
    ax.plot([0, A], [0, 0], color=color, lw=lw, solid_capstyle='round')
    ax.plot([0, 0], [0, D], color=color, lw=lw, solid_capstyle='round')
    ax.plot([A, A], [0, D], color=color, lw=lw, solid_capstyle='round')

    ax.scatter([mid - feed_gap / 2, mid + feed_gap / 2], [E, E], s=120, color=color, zorder=5)
    ax.text(mid, E - B * 0.55, 'Feed point', ha='center', va='center', fontsize=11, bbox=label_box)
    ax.text(mid, -E * 0.11, 'Reflector', ha='center', va='top', fontsize=11, bbox=label_box)

    arrowprops = dict(arrowstyle='<->', color='#4b0082', lw=1.4, shrinkA=0, shrinkB=0)
    ax.annotate('', xy=(0, E + E * 0.18), xytext=(A, E + E * 0.18), arrowprops=arrowprops)
    ax.text(mid, E + E * 0.21, f'A span = {A:.1f} {label}', ha='center', color='#4b0082', bbox=label_box)

    right_arrow_x = A + A * 0.10
    right_label_x = A + A * 0.14
    ax.annotate('', xy=(right_arrow_x, E), xytext=(right_arrow_x, E - B), arrowprops=arrowprops)
    ax.text(right_label_x, E - B / 2, f'B tail = {B:.1f}', va='center', color='#4b0082', bbox=label_box)

    ax.annotate('', xy=(right_arrow_x, E - B), xytext=(right_arrow_x, D), arrowprops=arrowprops)
    ax.text(right_label_x, (E - B + D) / 2, f'C gap = {C:.1f}', va='center', color='#4b0082', bbox=label_box)

    ax.annotate('', xy=(right_arrow_x, 0), xytext=(right_arrow_x, D), arrowprops=arrowprops)
    ax.text(right_label_x, D / 2, f'D tail = {D:.1f}', va='center', color='#4b0082', bbox=label_box)

    spacing_x = A * 0.72
    ax.annotate('', xy=(spacing_x, 0), xytext=(spacing_x, E), arrowprops=arrowprops)
    ax.text(spacing_x + A * 0.03, E * 0.68, f'E spacing = {E:.1f}', va='center', color='#4b0082', bbox=label_box)

    support_color = '#1f77b4'
    support_lw = 2.4
    ax.plot([mid - outer_pipe, mid + outer_pipe], [E, E], ':', color=support_color, lw=support_lw)
    ax.plot([mid - outer_pipe, mid + outer_pipe], [0, 0], ':', color=support_color, lw=support_lw)
    ax.plot([mid, mid], [(E - center_pipe) / 2, (E + center_pipe) / 2], ':', color=support_color, lw=support_lw)
    ax.text(mid, E + E * 0.32, f'4x outer pipe = {outer_pipe:.1f} {label}', ha='center', color=support_color, bbox=label_box)
    ax.text(mid - A * 0.20, E * 0.50, f'center pipe = {center_pipe:.1f} {label}', va='center', color=support_color, bbox=label_box)

    ax.plot([0, A], [E, 0], '--', color='#00a65a', lw=1.8)
    diagonal_label = dimensions['diagonal'] * scale
    ax.text(A * 0.24, E * 0.40, f'diagonal = {diagonal_label:.1f} {label}', color='#008a4a', bbox=label_box)

    ax.set_title(f'Moxon rectangle starter dimensions at {frequency_mhz:.3f} MHz', pad=18)
    ax.set_aspect('equal', adjustable='box')
    margin_x = A * 0.22
    margin_y = E * 0.40
    ax.set_xlim(-margin_x * 0.35, A + margin_x)
    ax.set_ylim(-margin_y, E + margin_y)
    ax.axis('off')
    fig.tight_layout()
    plt.show()

def show_moxon(frequency_mhz=435.0, conductor_diameter_mm=1.0, velocity_factor=1.0, units='mm'):
    dims = moxon_dimensions(frequency_mhz, conductor_diameter_mm, velocity_factor)
    display(HTML(f'<h3>Wavelength: {dims["wavelength"]:.4f} m</h3>'))
    plot_moxon(dims, frequency_mhz, units)
    display(format_geometry_table(dims, units))
    display(format_build_table(dims, units))



## Interactive Calculator

Enter the exact frequency and wire diameter. Useful starting points are `145 MHz` for 2 m and `435 MHz` for 70 cm.

In [ ]:
from IPython.display import display

frequency = widgets.BoundedFloatText(value=435.0, min=1.0, max=3000.0, description='Frequency MHz', step=0.025)
diameter = widgets.BoundedFloatText(value=1.0, min=0.1, max=50.0, description='Wire dia. mm', step=0.1)
velocity = widgets.FloatSlider(value=1.0, min=0.90, max=1.00, step=0.005, description='Velocity factor', continuous_update=False)
units = widgets.Dropdown(value='mm', options=['mm', 'cm', 'm'], description='Units')

presets = widgets.ToggleButtons(
    options=[('70 cm / 435 MHz', 435.0), ('2 m / 145 MHz', 145.0), ('Custom', None)],
    value=435.0,
    description='Preset',
)

def apply_preset(change):
    if change['new'] is not None:
        frequency.value = change['new']

presets.observe(apply_preset, names='value')

controls = widgets.VBox([presets, widgets.HBox([frequency, diameter, units]), velocity])
output = widgets.interactive_output(
    show_moxon,
    {
        'frequency_mhz': frequency,
        'conductor_diameter_mm': diameter,
        'velocity_factor': velocity,
        'units': units,
    },
)
display(widgets.VBox([controls, output]))

## Static Reference Values

These cells are useful when exporting the notebook to HTML or viewing it without interactive widget support.

In [ ]:
show_moxon(145.0, conductor_diameter_mm=1.0, velocity_factor=1.0, units='mm')

In [ ]:
show_moxon(435.0, conductor_diameter_mm=1.0, velocity_factor=1.0, units='mm')